In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
TRAIN_DIR = "/kaggle/input/datasets/saifurrahmanmamun/cifar-10/train"
VAL_DIR   = "/kaggle/input/datasets/saifurrahmanmamun/cifar-10/val"
TEST_DIR  = "/kaggle/input/datasets/saifurrahmanmamun/cifar-10/test"


train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    # validation_split=0.2,    # Automatically reserve 20% of images for testing
    # subset="training",       # Tell it to grab the 80% half
    # seed=42,                 # Ensures the 80/20 split is identical every time you run it
    image_size=(32, 32),     # CRUCIAL: Force all images to match your Conv2D Input layer shape
    batch_size=32
)
print(train_ds.class_names)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    # validation_split=0.2,
    # subset="validation",     # Tell it to grab the remaining 20% half
    # seed=42,
    image_size=(32, 32),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=(32,32),
    batch_size=32
)
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Found 45000 files belonging to 10 classes.


I0000 00:00:1783011270.978869      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783011270.982152      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Found 5000 files belonging to 10 classes.
Found 10000 files belonging to 10 classes.


In [3]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [4]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),

    
    layers.Flatten()
])

In [11]:
def extract_features_labels(dataset, model):
    features, labels = [], []
    for images, lbls in dataset:
        feat = model.predict(images, verbose=0)
        features.append(feat)
        labels.append(lbls.numpy())
    return np.concatenate(features), np.concatenate(labels)

In [12]:
X_train_features, y_train = extract_features_labels(train_ds, model)
X_val_features, y_val     = extract_features_labels(val_ds, model)
X_test_features, y_test   = extract_features_labels(test_ds, model)

In [13]:
print("Feature shapes:", X_train_features.shape, X_val_features.shape, X_test_features.shape)

Feature shapes: (45000, 1024) (5000, 1024) (10000, 1024)


In [14]:
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features)
X_test_features = scaler.transform(X_test_features)

In [ ]:
svm = SVC(kernel='rbf', C=1.0)
svm.fit(X_train_features, y_train)
svm_pred = svm.predict(X_test_features)
svm_acc = accuracy_score(y_test, svm_pred)
print("SVM Accuracy:", svm_acc)

In [ ]:
train_ds_features = model.predict(train_ds)

test_ds_features = model.predict(test_ds)

In [ ]:
xgb = XGBClassifier()

xgb.fit(train_ds_features, y_train)

xgb_pred = xgb.predict(test_ds_features)
accuracy_score(y_test, xgb_pred)

In [ ]:
rf = RandomForestClassifier()

rf.fit(train_ds_features, y_train)
rf_pred = rf.predict(test_ds_features)
accuracy_score(y_test, rf_pred)

In [ ]:
knn = KNeighborsClassifier()

knn.fit(train_ds_features, y_train)
knn_pred = knn.predict(test_ds_features)
accuracy_score(y_test, knn_pred)

In [ ]:
svm = SVC()

svm.fit(train_ds_features, y_train)
svm_pred = svm.predict(test_ds_features)
accuracy_score(y_test, svm_pred)

In [ ]:
lr = LogisticRegression(max_iter=5000)

lr.fit(train_ds_features, y_train)

lr_pred = lr.predict(test_ds_features)
accuracy_score(y_test, lr_pred)